# 43 · Images and PDFs as input
Level L4 · Part 5 · deep · ~30 min · builds on 12

## Goal
Send Scout a chart image and a small PDF together with a question, and estimate each one's token cost
before you send it, then check the estimate against what Gemini reports.

## The idea
A message is a list of **parts**. So far every part was text. A part can also hold raw bytes with a
**MIME type**, such as `image/png` or `application/pdf`. That is **multimodal input**. The model turns each
image or PDF page into **image tokens**, and you pay for them at the normal input price. On Gemini 3 models
the number does not depend on the file size or the pixel count. It is a fixed budget per image or per page,
set by **media resolution** (low, medium, high). So you can estimate the cost before the call.

Analogy: a photocopy shop that charges per page at a chosen quality, not per word on the page.

## Picture
```mermaid
flowchart LR
    F["chart.png / note.pdf<br/>(bytes)"] --> P["Part.from_bytes<br/>+ mime type + resolution"]
    Q["question (text)"] --> C["contents = [file part, text part]"]
    P --> C
    C --> E["estimate: table lookup<br/>(no call)"]
    C --> G["generate_content"]
    G --> U["usage by modality:<br/>IMAGE + TEXT"] --> D["$ for this turn"]
```

## Step by step
### 1. Start the recorder
Model calls are recorded once to `../data/llm_cassettes/43_multimodal-inputs.yaml` and replayed after that,
with no key (lesson 07).

In [1]:
import sys; sys.path.insert(0, "..")
import llm_replay
llm_replay.start("43_multimodal-inputs")

LLM mode: replay (43_multimodal-inputs.yaml)


### 2. Setup from lessons 07 and 12
The client, the pinned model, the price table, `cost_usd` and `usage_tokens`, all unchanged. Scout's notes
gain a seventh note that carries an **attachment**: a chart file instead of plain text.

In [2]:
import os, json, pathlib
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv("../.env")
MODEL = os.environ.get("TUT_LLM_MODEL", "gemini-3.5-flash-lite")
client = genai.Client(vertexai=True, api_key=os.environ.get("GOOGLE_CLOUD_API_KEY") or "replay-no-key-needed")
NO_AFC = types.AutomaticFunctionCallingConfig(disable=True)
PRICES_PER_MTOK = {"gemini-3.5-flash-lite": {"input": 0.30, "output": 2.50}}   # lesson 12, checked 2026-10-06

def cost_usd(model: str, input_tokens: int, output_tokens: int) -> float:
    price = PRICES_PER_MTOK[model]
    return (input_tokens * price["input"] + output_tokens * price["output"]) / 1_000_000

def usage_tokens(resp) -> tuple[int, int]:
    u = resp.usage_metadata
    return u.prompt_token_count or 0, (u.candidates_token_count or 0) + (u.thoughts_token_count or 0)

FILES = pathlib.Path("../data/scout_files_43"); FILES.mkdir(parents=True, exist_ok=True)
NOTE_7 = {"id": 7, "text": "Day lengths chart from my astronomy class.", "attachment": "day_lengths.png"}
print("model:", MODEL, "| note 7 attachment:", NOTE_7["attachment"])

model: gemini-3.5-flash-lite | note 7 attachment: day_lengths.png


### 3. Make the chart
We draw the chart ourselves with matplotlib, so the lesson needs no download. The numbers are planet day
lengths in hours. None of Scout's text notes mention Jupiter's or Saturn's day, so the model can only
answer from the picture.

In [3]:
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5), dpi=100)                    # 800 x 500 pixels
ax.bar(["Earth", "Mars", "Jupiter", "Saturn"], [23.9, 24.6, 9.9, 10.7], color="steelblue")
ax.set_ylabel("hours"); ax.set_title("Length of a day")
chart_path = FILES / NOTE_7["attachment"]
fig.savefig(chart_path, metadata={"Software": None})
chart_bytes = chart_path.read_bytes()
print(f"{chart_path.name}: {len(chart_bytes):,} bytes, 800 x 500 px")

day_lengths.png: 13,347 bytes, 800 x 500 px


### 4. The smallest multimodal call
`types.Part.from_bytes(data=..., mime_type=...)` wraps the bytes in a part. `contents` becomes a list:
the image first, then the question. Google and Anthropic both suggest putting the file before the question.

In [4]:
question = "Which planet in this chart has the shortest day, and about how many hours is it? One sentence."
image_part = types.Part.from_bytes(data=chart_bytes, mime_type="image/png")
config = types.GenerateContentConfig(max_output_tokens=80, automatic_function_calling=NO_AFC)
resp = client.models.generate_content(model=MODEL, contents=[image_part, question], config=config)
print("Scout:", resp.text.strip())
REC = {"image": {"reply": resp.text.strip(), "usage": list(usage_tokens(resp)),
                 "by_modality": {d.modality.name: d.token_count for d in resp.usage_metadata.prompt_tokens_details}}}

Scout: Based on the chart, **Jupiter** has the shortest day, which is about **10 hours** long.


If you skipped the model cell, this loads the recorded result of that call so the rest still runs.

In [5]:
RECORDED = pathlib.Path("../data/recorded/43.json")
if "REC" not in globals():   # recorded output of step 4 (and later steps), saved on a real run
    REC = json.loads(RECORDED.read_text())
print("reply:", REC["image"]["reply"])
print("input, output tokens:", REC["image"]["usage"], "| input by modality:", REC["image"]["by_modality"])

reply: Based on the chart, **Jupiter** has the shortest day, which is about **10 hours** long.
input, output tokens: [1113, 23] | input by modality: {'TEXT': 21, 'IMAGE': 1092}


Scout read the bars correctly. Look at the input split: the image cost 1,092 tokens, the question 21.
`prompt_tokens_details` breaks the input down by **modality**, so you can see where the money went.

### 5. What does that cost?
Image tokens are billed as ordinary input tokens. Same price table, same `cost_usd` as lesson 12.

In [6]:
tokens_in, tokens_out = REC["image"]["usage"]
image_tokens = REC["image"]["by_modality"]["IMAGE"]
print(f"whole turn: ${cost_usd(MODEL, tokens_in, tokens_out):.6f}")
print(f"the image alone: ${cost_usd(MODEL, image_tokens, 0):.6f}  "
      f"-> ${cost_usd(MODEL, image_tokens, 0) * 1000:.2f} per 1,000 images")

whole turn: $0.000391
the image alone: $0.000328  -> $0.33 per 1,000 images


One chart is a fraction of a cent. A thousand of them, or a chat that resends the same chart on every turn
(lesson 12's history rule applies to images too), adds up quickly.

### 6. Estimate before sending
For Gemini 3 models Google publishes a fixed token budget per image and per PDF page for each media
resolution. The default ("unspecified") is 1,120 per image and 560 per PDF page. A lookup table is enough.

In [7]:
GEMINI3_MEDIA_TOKENS = {   # ai.google.dev/gemini-api/docs/media-resolution, checked 2026-10-11
    "image": {"unspecified": 1120, "low": 280, "medium": 560, "high": 1120, "ultra_high": 2240},
    "pdf_page": {"unspecified": 560, "low": 280, "medium": 560, "high": 1120},
}

def estimate_media_tokens(kind: str, count: int = 1, resolution: str = "unspecified") -> int:
    """kind is 'image' or 'pdf_page'; count is images or pages. Native PDF text is not billed on Gemini 3 (step 9), so it is not added."""
    return GEMINI3_MEDIA_TOKENS[kind][resolution] * count

print("our 800x500 chart, default:", estimate_media_tokens("image"))
print("a 4K photo, default:       ", estimate_media_tokens("image"), "(same: pixels do not matter)")

our 800x500 chart, default: 1120
a 4K photo, default:        1120 (same: pixels do not matter)


Other vendors count differently. Claude splits the image into 28 x 28 pixel patches, so its cost grows with
the pixel count up to a cap (1,568 on standard models, 4,784 on Claude 4.7 and later, after downscaling).
A quick comparison for our chart, without downscaling:

In [8]:
import math

def claude_image_tokens(width: int, height: int, cap: int = 1568) -> int:
    """platform.claude.com vision docs: ceil(w/28) * ceil(h/28); bigger images are downscaled to the cap."""
    return min(math.ceil(width / 28) * math.ceil(height / 28), cap)

for w, h in [(200, 200), (800, 500), (1000, 1000)]:
    print(f"{w}x{h}: Claude ~{claude_image_tokens(w, h)} tokens | Gemini 3 {estimate_media_tokens('image')}")

200x200: Claude ~64 tokens | Gemini 3 1120
800x500: Claude ~522 tokens | Gemini 3 1120
1000x1000: Claude ~1296 tokens | Gemini 3 1120


So a tiny icon is cheap on Claude and still 1,120 tokens on Gemini 3 at default resolution. Always estimate
with the formula of the vendor you call, and confirm with its own counter.

### 7. Confirm with `count_tokens`
Lesson 12's free counter accepts image parts too. We count the chart at three resolutions, set per part
with `media_resolution`.

In [9]:
counted = {}
for level in ["unspecified", "low", "high"]:
    res = types.PartMediaResolution(level=f"MEDIA_RESOLUTION_{level.upper()}")
    part = types.Part.from_bytes(data=chart_bytes, mime_type="image/png", media_resolution=res)
    counted[level] = client.models.count_tokens(model=MODEL, contents=[part]).total_tokens
REC["counted"] = counted
print(counted)

{'unspecified': 1120, 'low': 280, 'high': 1120}


Compare the counter with the table (recorded counts are used if you skipped the cell).

In [10]:
counted = REC.get("counted") or json.loads(RECORDED.read_text())["counted"]
for level, n in counted.items():
    est = estimate_media_tokens("image", resolution=level)
    print(f"{level:>11}: estimate {est:>5} | count_tokens {n:>5} | {'match' if est == n else 'DIFFERENT'}")

unspecified: estimate  1120 | count_tokens  1120 | match
        low: estimate   280 | count_tokens   280 | match
       high: estimate  1120 | count_tokens  1120 | match


The table matches Gemini's counter exactly. Low resolution costs a quarter of the default.
In all three of our billed calls the bill was a little lower than the count (1,092 vs 1,120; 273 vs 280;
1,054 vs 1,120). Treat the table and `count_tokens` as a planning estimate that, in our runs, never
under-counted. The Gemini docs don't say it is a guaranteed ceiling, so budget from `usage` after the call.

### 8. Is low resolution good enough?
Cheaper only helps if the answer is still right. Ask the same question with the chart at low resolution.

In [11]:
low_part = types.Part.from_bytes(data=chart_bytes, mime_type="image/png",
                                 media_resolution=types.PartMediaResolution(level="MEDIA_RESOLUTION_LOW"))
resp_low = client.models.generate_content(model=MODEL, contents=[low_part, question], config=config)
REC["image_low"] = {"reply": resp_low.text.strip(), "usage": list(usage_tokens(resp_low))}
print("Scout (low):", REC["image_low"]["reply"], "| tokens:", REC["image_low"]["usage"])

Scout (low): Based on the chart, Jupiter has the shortest day, which is about 10 hours long. | tokens: [294, 20]


Compare the two runs side by side (recorded values if you skipped the cell).

In [12]:
if "image_low" not in REC:   # recorded output of step 8
    REC["image_low"] = json.loads(RECORDED.read_text())["image_low"]
for name in ["image", "image_low"]:
    tin, tout = REC[name]["usage"]
    print(f"{name:>9}: {tin:>5} in, ${cost_usd(MODEL, tin, tout):.6f} | {REC[name]['reply']}")

    image:  1113 in, $0.000391 | Based on the chart, **Jupiter** has the shortest day, which is about **10 hours** long.
image_low:   294 in, $0.000138 | Based on the chart, Jupiter has the shortest day, which is about 10 hours long.


For a simple bar chart with big labels, low resolution gave the same answer with about a quarter of the
input tokens and about a third of the cost (output tokens did not shrink).
Dense pages, small print or screenshots may need high. Test on your own files before choosing.

### 9. A PDF as input
A PDF goes in the same way, with `mime_type="application/pdf"`. We build a two-page PDF: page 1 is a text
note, page 2 is the chart. matplotlib writes real PDF text, so page 1 has **native text** the model can read.

In [13]:
from matplotlib.backends.backend_pdf import PdfPages

pdf_path = FILES / "astronomy_notes.pdf"
with PdfPages(pdf_path, metadata={"CreationDate": None}) as pdf:
    page1 = plt.figure(figsize=(8.27, 11.69))                       # A4 portrait
    page1.text(0.1, 0.9, "Astronomy class notes\n\nJupiter: a day lasts about 10 hours.\n"
               "Mars: a day lasts about 24 hours and 39 minutes.\nThe chart is on the next page.",
               va="top", fontsize=14)
    pdf.savefig(page1); pdf.savefig(fig)
pdf_bytes = pdf_path.read_bytes()
print(f"{pdf_path.name}: {len(pdf_bytes):,} bytes, 2 pages | estimate: {estimate_media_tokens('pdf_page', 2)} tokens")

astronomy_notes.pdf: 14,996 bytes, 2 pages | estimate: 1120 tokens


Now ask a question that needs both pages, and count the tokens first.

In [14]:
pdf_part = types.Part.from_bytes(data=pdf_bytes, mime_type="application/pdf")
pdf_question = "Does the text on page 1 agree with the chart on page 2 about Jupiter's day? One sentence."
REC["pdf_counted"] = client.models.count_tokens(model=MODEL, contents=[pdf_part]).total_tokens
resp_pdf = client.models.generate_content(model=MODEL, contents=[pdf_part, pdf_question], config=config)
REC["pdf"] = {"reply": resp_pdf.text.strip(), "usage": list(usage_tokens(resp_pdf)),
              "by_modality": {d.modality.name: d.token_count for d in resp_pdf.usage_metadata.prompt_tokens_details}}
RECORDED.write_text(json.dumps(REC, indent=1))       # saved for the fallbacks and the solutions
print("Scout:", REC["pdf"]["reply"])

Scout: Yes, the text on page 1 states that a day on Jupiter lasts about 10 hours, which matches the bar height of approximately 10 hours shown for Jupiter on the chart on page 2.


The PDF result, from the live run or the recording:

In [15]:
if "pdf" not in REC:   # recorded output of step 9
    REC.update({k: json.loads(RECORDED.read_text())[k] for k in ["pdf", "pdf_counted"]})
print("reply:", REC["pdf"]["reply"])
print("count_tokens (file only):", REC["pdf_counted"], "| estimate:", estimate_media_tokens("pdf_page", 2))
print("usage by modality:", REC["pdf"]["by_modality"], "| total in, out:", REC["pdf"]["usage"])

reply: Yes, the text on page 1 states that a day on Jupiter lasts about 10 hours, which matches the bar height of approximately 10 hours shown for Jupiter on the chart on page 2.
count_tokens (file only): 1120 | estimate: 1120
usage by modality: {'IMAGE': 1054, 'TEXT': 24} | total in, out: [1078, 43]


Scout combined the text page and the chart page. `count_tokens` matches the table: 2 pages x 560 = 1,120.
The bill was again a bit lower (1,054). The pages appear under `IMAGE`, because each page is read as an
image. (`count_tokens` reports the same file under the modality `DOCUMENT`, while billed usage reports it
as `IMAGE`.) Google's docs say native PDF text is extracted and **not billed** on Gemini 3. Our usage is
consistent with that: `TEXT` is only 24 tokens, about the size of the question, and the total input (1,078)
is below the 1,120 page budget. We did not run a text-free control, so this is evidence, not proof.

### 10. Scout checks the cost of an attachment before reading it
Scout's new helper: pick the MIME type from the file name, estimate the cost, refuse if it is over budget,
otherwise send. `mimetypes` from the standard library maps extensions to MIME types.

In [16]:
import mimetypes

def attachment_estimate(path: pathlib.Path, pages: int = 1, resolution: str = "unspecified") -> int:
    mime = mimetypes.guess_type(path.name)[0]
    kind = "pdf_page" if mime == "application/pdf" else "image"
    return estimate_media_tokens(kind, pages, resolution)

def ask_about_attachment(note: dict, question: str, budget_usd: float, pages: int = 1, max_out: int = 80):
    path = FILES / note["attachment"]
    worst = cost_usd(MODEL, attachment_estimate(path, pages) + 100, max_out)   # +100 for the question text
    if worst > budget_usd:
        return f"refused: worst case ${worst:.6f} is over the ${budget_usd} budget"
    part = types.Part.from_bytes(data=path.read_bytes(), mime_type=mimetypes.guess_type(path.name)[0])
    cfg = types.GenerateContentConfig(max_output_tokens=max_out, automatic_function_calling=NO_AFC)
    return client.models.generate_content(model=MODEL, contents=[part, question], config=cfg).text.strip()

A tight budget stops the call before any money is spent. No model is needed to see that.

In [17]:
print(ask_about_attachment(NOTE_7, question, budget_usd=0.0002))
print("worst case at default resolution:",
      f"${cost_usd(MODEL, attachment_estimate(chart_path) + 100, 80):.6f}")

refused: worst case $0.000566 is over the $0.0002 budget
worst case at default resolution: $0.000566


With a budget that fits, Scout answers from the attachment.

In [18]:
answer = ask_about_attachment(NOTE_7, "Which planet has the longest day in this chart? One sentence.", 0.001)
print("Scout:", answer)

Scout: Based on the chart, **Mars** has the longest day, with a bar height just under 25 hours.


## At work
- **Images in a chat are resent and rebilled every turn,** like any history (lesson 12). Teams upload large
  files once to a Files API and reference them by id, which keeps requests small; caching and trimming
  old attachments cut the token bill. → lesson 54
- **Resolution is a cost knob.** Claude 4.7+ models can use about three times the visual tokens of older
  ones for the same image, and Gemini 3 charges 4x more at default than at low. Pipelines downscale or pick
  a resolution per document type and log tokens per modality from `usage`. → lessons 49 and 54
- **Text inside an image or PDF is untrusted input.** A scanned page can carry instructions just as a web
  page can; the same defences apply. → lesson 51
- **Bulk document jobs are planned with per-page estimates** (pages x tokens per page x price) before they
  run, then sized against rate limits measured in tokens per minute. → lesson 56

## What just happened
- `types.Part.from_bytes(data, mime_type)` put a PNG and a PDF into `contents`, next to the question.
- Scout answered "which planet has the shortest day" from the chart alone, and compared a PDF's text page
  with its chart page.
- `prompt_tokens_details` split the input into `IMAGE` and `TEXT`: 1,092 image tokens, 21 for the question.
- A lookup table of Gemini 3 media-resolution budgets matched `count_tokens` exactly (1,120 / 280 / 1,120);
  in our runs the bill was at or just below both.
- Low resolution answered the chart question correctly with about a quarter of the input tokens.
- The two-page PDF counted 2 x 560 = 1,120 tokens (1,054 IMAGE tokens billed); the TEXT count (24) is in line
  with the question alone, consistent with native text not being billed.
- `ask_about_attachment` refused an over-budget file before calling the model.

## Common mistakes
**1. Passing the file path instead of the bytes.** `data` must be the file's bytes. A string is read as
base64, so a path fails validation.

In [19]:
types.Part.from_bytes(data=str(chart_path), mime_type="image/png")

ValidationError: 1 validation error for Blob
data
  Data should be valid base64: Invalid symbol 46, offset 0. [type=bytes_invalid_encoding, input_value='../data/scout_files_43/day_lengths.png', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/bytes_invalid_encoding

Fix: `types.Part.from_bytes(data=chart_path.read_bytes(), mime_type="image/png")`.

**2. Estimating image cost from the file size.** Someone divides the bytes by 4, as for text. For our
chart that is wildly off in one direction, and for a tiny PNG it would be off in the other.

In [20]:
guess = len(chart_bytes) // 4
real = counted["unspecified"]
assert abs(guess - real) / real < 0.5, f"bytes/4 guessed {guess} tokens, Gemini counted {real}"

AssertionError: bytes/4 guessed 3336 tokens, Gemini counted 1120

Fix: images and pages have their own rule per vendor and model. Use the published table
(`estimate_media_tokens`) and confirm with `count_tokens`.

## Try it
Reuse the saved results in `../data/recorded/43.json`; no new model calls needed.
1. A support bot gets 5,000 screenshots a day. Using `estimate_media_tokens` and `cost_usd`, what is the
   daily input cost for the images at default resolution, and at low resolution?
2. Write `pdf_estimate(pages, resolution)` that returns the input tokens and dollars for a PDF on `MODEL`.
   How much does a 300-page report cost at `"medium"`? (Ignore native text, which is not billed.)

In [21]:
# your code here

Finally, stop the recorder (in record mode this saves the recording).

In [22]:
llm_replay.stop()

## Key terms
- **multimodal input** — a request whose parts include images, PDFs, audio or video, not only text; each part is bytes plus a MIME type.
- **MIME type** — a label for a file's format, such as `image/png` or `application/pdf`, sent with the bytes so the model knows how to read them.
- **image tokens** — the input tokens a model uses to represent an image or a PDF page; billed at the normal input price.
- **media resolution** — Gemini 3's per-part setting (low, medium, high, ultra high) that fixes how many tokens an image or page gets.
- **PDF input** — sending a PDF as a part; each page is read as an image, and native text in it is extracted (not billed on Gemini 3, per Google's docs).
- **tokens by modality** — the `prompt_tokens_details` breakdown in Gemini's usage, splitting input into TEXT, IMAGE and other kinds.